# 📈 Stock Price Prediction Using Machine Learning
## Google Colab Notebook — Complete Pipeline

**Models:** LSTM · Random Forest · Linear Regression  
**Data:** Yahoo Finance (yfinance) — AAPL default, ~5 years daily  
**Environment:** Google Colab (GPU recommended)

---

### ⚠️ Disclaimer
> This is an **educational project** only. Predictions are not investment advice.
> Market prices are inherently uncertain. Past patterns do not guarantee future results.

---

### How to run
1. Set runtime to **GPU**: Runtime → Change runtime type → GPU  
2. (Optional) Mount Google Drive in **Section 0** to persist files  
3. **Runtime → Run all** (Ctrl+F9)  
4. The notebook downloads data automatically — no manual CSV upload needed

**Estimated runtime:** ~10–20 min (GPU) · ~25–40 min (CPU)

---
## Section 0 — Optional: Mount Google Drive
Run this cell to save all outputs to your Google Drive (survives session disconnects).  
**Skip this cell** to use Colab's temporary local storage instead.

In [ ]:
# ============================================================
# OPTIONAL: Mount Google Drive
# Comment out this entire cell if you prefer local Colab storage.
# ============================================================
USE_DRIVE = False  # Set to True to enable Drive mounting

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/stock-price-prediction'
    print(f'Using Google Drive: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '/content/stock-price-prediction'
    print(f'Using local Colab storage: {PROJECT_ROOT}')

import os
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root ready.')

---
## Section 1 — Environment Setup & Library Installation

In [ ]:
# Install / upgrade required libraries
# (Colab already has TensorFlow and scikit-learn, but we pin versions)
import subprocess, sys

packages = [
    'yfinance>=0.2.28',
    'scikit-learn>=1.3',
    'tensorflow>=2.13',
    'matplotlib>=3.7',
    'seaborn>=0.12',
    'joblib>=1.3',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed.')

In [ ]:
# ============================================================
# Global imports and configuration
# ============================================================
import os
import sys
import json
import random
import logging
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# ============================================================
# Configuration — edit these to change behaviour
# ============================================================
TICKER          = 'AAPL'       # Stock ticker symbol
YEARS_HISTORY   = 5            # Years of historical data
LOOK_BACK       = 60           # LSTM sequence length (trading days)
N_LAG_CLOSE     = 5            # Lag Close values for RF/LR
TRAIN_RATIO     = 0.70         # 70% training
VAL_RATIO       = 0.10         # 10% validation
# TEST_RATIO    = 0.20         # 20% test (remainder)
RANDOM_SEED     = 42
BATCH_SIZE      = 32
MAX_EPOCHS      = 50
PATIENCE        = 10
RF_ESTIMATORS   = 200

# Directory layout
try:
    # If PROJECT_ROOT was set in Section 0, use it; otherwise default
    _root = PROJECT_ROOT
except NameError:
    _root = '/content/stock-price-prediction'
    os.makedirs(_root, exist_ok=True)

DATA_DIR        = os.path.join(_root, 'data')
RAW_DIR         = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR   = os.path.join(DATA_DIR, 'processed')
ARTIFACT_DIR    = os.path.join(_root, 'artifacts', TICKER)
FIGURES_DIR     = os.path.join(_root, 'reports', 'figures')

for d in [RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Fix random seeds for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('='*60)
print(f'  Stock Price Prediction Pipeline')
print(f'  Ticker     : {TICKER}')
print(f'  History    : {YEARS_HISTORY} years')
print(f'  Look-back  : {LOOK_BACK} trading days')
print(f'  Split      : {int(TRAIN_RATIO*100)}% / {int(VAL_RATIO*100)}% / 20%')
print(f'  Artifact dir: {ARTIFACT_DIR}')
print('='*60)
print(f'TensorFlow version  : {tf.__version__}')
print(f'scikit-learn version: {__import__("sklearn").__version__}')
print(f'yfinance version    : {yf.__version__}')

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus[0].name}')
else:
    print('No GPU detected — training on CPU (slower).')

---
## Section 2 — Automatic Historical Stock Data Collection

Downloads ~5 years of daily OHLCV data via `yfinance`.  
- Dynamic date range (no hardcoded year)
- Handles MultiIndex columns from some yfinance versions
- Saves raw CSV to `data/raw/`
- No artificial weekend/holiday rows

In [ ]:
# ============================================================
# Data download utilities (inline — no separate file needed)
# ============================================================

REQUIRED_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']

def download_stock_data(ticker, years=5, save_dir=RAW_DIR, interval='1d'):
    """
    Download historical OHLCV data via yfinance.
    
    - Dynamic 5-year range from today
    - Handles MultiIndex columns
    - Removes duplicates, sorts ascending
    - Saves raw CSV
    - Returns None on failure
    """
    ticker = ticker.strip().upper()
    end_date   = datetime.today()
    start_date = end_date - timedelta(days=years * 365 + 10)
    
    print(f'Downloading {ticker} | {start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")} | interval={interval}')
    
    try:
        raw = yf.download(
            ticker,
            start=start_date.strftime('%Y-%m-%d'),
            end=end_date.strftime('%Y-%m-%d'),
            interval=interval,
            auto_adjust=True,
            progress=False,
        )
    except Exception as e:
        print(f'ERROR: Download failed — {e}')
        return None
    
    if raw is None or raw.empty:
        print(f"ERROR: No data returned for '{ticker}'. Check ticker and internet.")
        return None
    
    # Handle MultiIndex columns (some yfinance versions)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    
    # Normalise to title-case
    raw.columns = [str(c).strip().title() for c in raw.columns]
    
    # Ensure Date index
    if 'Date' in raw.columns:
        raw = raw.set_index('Date')
    raw.index.name = 'Date'
    
    # Check required columns
    missing = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
    if missing:
        print(f'ERROR: Missing columns: {missing}. Available: {list(raw.columns)}')
        return None
    
    df = raw[REQUIRED_COLUMNS].copy()
    df.sort_index(inplace=True)
    
    # Remove duplicate dates
    n_dupes = df.index.duplicated().sum()
    if n_dupes:
        print(f'  Removing {n_dupes} duplicate date(s)')
        df = df[~df.index.duplicated(keep='last')]
    
    # Convert to numeric
    for col in REQUIRED_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Drop NaN rows
    n_nan = df.isnull().sum().sum()
    if n_nan:
        print(f'  Dropping {n_nan} NaN value(s)')
        df.dropna(inplace=True)
    
    if len(df) < 100:
        print(f'ERROR: Only {len(df)} rows — insufficient data.')
        return None
    
    # Save raw CSV
    os.makedirs(save_dir, exist_ok=True)
    csv_path = os.path.join(save_dir, f'{ticker}_daily.csv')
    df.to_csv(csv_path)
    
    print(f'  Downloaded {len(df)} rows  |  {df.index[0].date()} → {df.index[-1].date()}')
    print(f'  Raw CSV saved → {csv_path}')
    return df


print('Data download functions defined.')

In [ ]:
# ============================================================
# Download AAPL data
# ============================================================
df_raw = download_stock_data(TICKER, years=YEARS_HISTORY, save_dir=RAW_DIR)

if df_raw is None:
    raise RuntimeError(
        f"Failed to download data for '{TICKER}'. "
        "Check ticker symbol and internet connection."
    )

print(f'\nSuccessfully downloaded {len(df_raw)} trading days of {TICKER} data.')

---
## Section 3 — Data Validation & Exploratory Data Analysis

In [ ]:
# ============================================================
# Data validation
# ============================================================
print('=== DATA VALIDATION ===')
print(f'Shape          : {df_raw.shape}')
print(f'Date range     : {df_raw.index[0].date()} to {df_raw.index[-1].date()}')
print(f'Columns        : {list(df_raw.columns)}')
print(f'Duplicated dates: {df_raw.index.duplicated().sum()}')
print(f'Missing values : {df_raw.isnull().sum().sum()}')
print(f'Sorted ascending: {df_raw.index.is_monotonic_increasing}')
print()
print('Data types:')
print(df_raw.dtypes)
print()
print('Summary statistics:')
display(df_raw.describe().round(2))

In [ ]:
# ============================================================
# Exploratory Data Analysis (EDA)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'{TICKER} — Exploratory Data Analysis ({df_raw.index[0].year}–{df_raw.index[-1].year})', fontsize=16, fontweight='bold')

# 1. Closing price history
axes[0, 0].plot(df_raw.index, df_raw['Close'], color='#2196F3', linewidth=1)
axes[0, 0].set_title('Closing Price History')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 2. Volume
axes[0, 1].bar(df_raw.index, df_raw['Volume'], color='#FF9800', alpha=0.6, width=1)
axes[0, 1].set_title('Daily Trading Volume')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Volume')
axes[0, 1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 3. Daily returns histogram
daily_ret = df_raw['Close'].pct_change().dropna() * 100
axes[1, 0].hist(daily_ret, bins=80, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(daily_ret.mean(), color='red', linestyle='--', label=f'Mean={daily_ret.mean():.2f}%')
axes[1, 0].set_title('Daily Return Distribution')
axes[1, 0].set_xlabel('Daily Return (%)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# 4. Rolling 30-day volatility
rolling_std = daily_ret.rolling(30).std()
axes[1, 1].plot(df_raw.index[1:], rolling_std, color='#9C27B0', linewidth=1)
axes[1, 1].set_title('30-Day Rolling Volatility (Std of Daily Return)')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Std Dev (%)')
axes[1, 1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
eda_path = os.path.join(FIGURES_DIR, 'eda_overview.png')
plt.savefig(eda_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'EDA chart saved → {eda_path}')

---
## Section 4 — Data Cleaning & Preprocessing

In [ ]:
# ============================================================
# Data cleaning summary
# ============================================================
# The download function already:
#   - Removed duplicates
#   - Sorted ascending
#   - Dropped NaN rows
#   - Normalised column names
# Here we verify the clean state:

assert df_raw.index.is_monotonic_increasing, 'Data not sorted'
assert not df_raw.index.duplicated().any(), 'Duplicate dates exist'
assert not df_raw.isnull().any().any(), 'NaN values exist'

print('Data cleaning checks passed:')
print(f'  {len(df_raw)} clean trading-day observations')
print(f'  Date range: {df_raw.index[0].date()} to {df_raw.index[-1].date()}')
print(f'  No fabricated weekend or holiday rows.')
print(f'  Price convention: split/dividend-adjusted (auto_adjust=True in yfinance)')

# Show first/last rows
display(pd.concat([df_raw.head(3), df_raw.tail(3)]))

---
## Section 5 — Technical Indicator Calculation & Feature Engineering

Features computed (all use **only current and past data**):
| Feature | Definition |
|---------|----------|
| MA5 | 5-day simple moving average of Close |
| MA20 | 20-day simple moving average of Close |
| RSI | 14-period RSI (Wilder's EMA smoothing) |
| BB_Upper / BB_Lower | 20-day Bollinger Bands, 2σ |
| Daily_Return | Percentage change of Close |
| Price_to_Volume_Ratio | Close / (Volume + 1) — guards zero volume |

Warm-up rows where rolling features are `NaN` are **dropped** (not filled).

In [ ]:
# ============================================================
# Feature engineering functions (inline)
# ============================================================

MA_SHORT  = 5
MA_LONG   = 20
RSI_PERIOD = 14
BB_WINDOW  = 20
BB_STD     = 2

FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'MA5', 'MA20', 'RSI', 'BB_Upper', 'BB_Lower',
    'Daily_Return', 'Price_to_Volume_Ratio'
]

def compute_rsi(series, period=RSI_PERIOD):
    """14-period RSI using Wilder's exponential smoothing."""
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def add_technical_indicators(df):
    """
    Add all technical indicator features to an OHLCV DataFrame.
    Drops warm-up NaN rows. Never fills fabricated values.
    """
    df = df.copy()
    close  = df['Close']
    volume = df['Volume']
    
    df['MA5']   = close.rolling(MA_SHORT).mean()
    df['MA20']  = close.rolling(MA_LONG).mean()
    df['RSI']   = compute_rsi(close, RSI_PERIOD)
    
    # Bollinger Bands
    bb_mid     = close.rolling(BB_WINDOW).mean()
    bb_std     = close.rolling(BB_WINDOW).std()
    df['BB_Upper'] = bb_mid + BB_STD * bb_std
    df['BB_Lower'] = bb_mid - BB_STD * bb_std
    
    df['Daily_Return']         = close.pct_change() * 100  # in percent
    df['Price_to_Volume_Ratio'] = close / (volume + 1)     # guard zero vol
    
    # Drop warm-up rows (no fill)
    rows_before = len(df)
    df.dropna(inplace=True)
    print(f'  Dropped {rows_before - len(df)} warm-up row(s) with NaN features.')
    print(f'  Remaining rows after feature engineering: {len(df)}')
    return df

print('Feature engineering functions defined.')
print(f'Feature columns ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}')

In [ ]:
# ============================================================
# Compute features
# ============================================================
print('Computing technical indicators...')
df_feat = add_technical_indicators(df_raw)

# Save processed dataset
processed_path = os.path.join(PROCESSED_DIR, f'{TICKER}_features.csv')
df_feat.to_csv(processed_path)
print(f'Processed dataset saved → {processed_path}')

print()
print('Feature sample (last 5 rows):')
display(df_feat[FEATURE_COLUMNS].tail(5).round(4))

In [ ]:
# ============================================================
# Visualise technical indicators
# ============================================================
fig, axes = plt.subplots(4, 1, figsize=(16, 18), sharex=True)
fig.suptitle(f'{TICKER} — Technical Indicators', fontsize=16, fontweight='bold')

# Subset last 2 years for readability
recent = df_feat.iloc[-504:]  # approx 2 years

# 1. Close + MA5 + MA20 + Bollinger Bands
axes[0].plot(recent.index, recent['Close'], label='Close', color='#2196F3', lw=1.5)
axes[0].plot(recent.index, recent['MA5'],   label='MA5',   color='#FF9800', lw=1, ls='--')
axes[0].plot(recent.index, recent['MA20'],  label='MA20',  color='#F44336', lw=1, ls='--')
axes[0].fill_between(recent.index, recent['BB_Lower'], recent['BB_Upper'], alpha=0.15, color='grey', label='Bollinger Bands')
axes[0].plot(recent.index, recent['BB_Upper'], color='grey', lw=0.8, ls=':')
axes[0].plot(recent.index, recent['BB_Lower'], color='grey', lw=0.8, ls=':')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].set_title('Close Price with Moving Averages & Bollinger Bands')

# 2. RSI
axes[1].plot(recent.index, recent['RSI'], color='#9C27B0', lw=1.2)
axes[1].axhline(70, color='red',   ls='--', lw=1, label='Overbought (70)')
axes[1].axhline(30, color='green', ls='--', lw=1, label='Oversold (30)')
axes[1].fill_between(recent.index, 30, 70, alpha=0.1, color='grey')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper left', fontsize=9)
axes[1].set_title(f'RSI ({RSI_PERIOD}-period)')

# 3. Daily Return
colors = ['#4CAF50' if r >= 0 else '#F44336' for r in recent['Daily_Return']]
axes[2].bar(recent.index, recent['Daily_Return'], color=colors, alpha=0.8, width=1)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_ylabel('Return (%)')
axes[2].set_title('Daily Return (%)')

# 4. Volume
axes[3].bar(recent.index, recent['Volume'], color='#607D8B', alpha=0.7, width=1)
axes[3].set_ylabel('Volume')
axes[3].set_xlabel('Date')
axes[3].set_title('Trading Volume')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
tech_path = os.path.join(FIGURES_DIR, 'technical_indicators.png')
plt.savefig(tech_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Technical indicator chart saved → {tech_path}')

---
## Section 6 — Chronological Train / Validation / Test Split

**Leakage-safe split (DECISIONS.md D7):**
- 70% training → 10% validation → 20% test
- No shuffling
- Scalers fitted on **training data only**, then applied to val/test
- Dedicated `close_scaler` for LSTM target inverse-transformation

In [ ]:
# ============================================================
# Chronological split
# ============================================================
n = len(df_feat)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

df_train = df_feat.iloc[:train_end].copy()
df_val   = df_feat.iloc[train_end:val_end].copy()
df_test  = df_feat.iloc[val_end:].copy()

print('Chronological split:')
print(f'  Train : {len(df_train)} rows  {df_train.index[0].date()} → {df_train.index[-1].date()}')
print(f'  Val   : {len(df_val)} rows  {df_val.index[0].date()} → {df_val.index[-1].date()}')
print(f'  Test  : {len(df_test)} rows  {df_test.index[0].date()} → {df_test.index[-1].date()}')
print()

# Verify no overlap
assert df_train.index[-1] < df_val.index[0],  'Train/val overlap!'
assert df_val.index[-1]   < df_test.index[0], 'Val/test overlap!'
print('No overlap between splits — chronological order confirmed.')

In [ ]:
# ============================================================
# Fit scalers — TRAINING DATA ONLY
# ============================================================

# Feature scaler (all 12 features)
feature_scaler = MinMaxScaler(feature_range=(0, 1))
feature_scaler.fit(df_train[FEATURE_COLUMNS].values)

# Dedicated Close scaler (for LSTM target inverse-transformation)
close_scaler = MinMaxScaler(feature_range=(0, 1))
close_scaler.fit(df_train[['Close']].values)

# Scale all partitions (NEVER refit on val/test)
scaled_train = feature_scaler.transform(df_train[FEATURE_COLUMNS].values)
scaled_val   = feature_scaler.transform(df_val[FEATURE_COLUMNS].values)
scaled_test  = feature_scaler.transform(df_test[FEATURE_COLUMNS].values)

print(f'Scalers fitted on training data ({len(df_train)} rows).')
print(f'Scaled train range: [{scaled_train.min():.4f}, {scaled_train.max():.4f}]')
print(f'Val/test may exceed [0,1] — scaler NOT refitted on them (correct behaviour).')

# CLOSE INDEX in feature list (for legacy reference)
CLOSE_IDX = FEATURE_COLUMNS.index('Close')
print(f'Close column index in feature list: {CLOSE_IDX}')
print(f'Dedicated close_scaler used for inverse-transform (not index-based).')

In [ ]:
# ============================================================
# LSTM sequence builder
# ============================================================

def build_lstm_sequences(scaled_features, close_scaler, raw_close, look_back=LOOK_BACK):
    """
    Build LSTM input/target sequences.
    X[i] = scaled_features[i : i+look_back]  (look_back x n_features)
    y[i] = scaled Close at index i+look_back
    No future data leaks into inputs.
    """
    scaled_close = close_scaler.transform(raw_close.reshape(-1, 1)).flatten()
    X, y = [], []
    for i in range(look_back, len(scaled_features)):
        X.append(scaled_features[i - look_back : i])  # past look_back steps
        y.append(scaled_close[i])                       # next-day target
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Build sequences
X_train_lstm, y_train_lstm = build_lstm_sequences(scaled_train, close_scaler, df_train['Close'].values)
X_val_lstm,   y_val_lstm   = build_lstm_sequences(scaled_val,   close_scaler, df_val['Close'].values)
X_test_lstm,  y_test_lstm  = build_lstm_sequences(scaled_test,  close_scaler, df_test['Close'].values)

n_features = len(FEATURE_COLUMNS)
print(f'LSTM sequence shapes:')
print(f'  X_train: {X_train_lstm.shape}  (samples, look_back={LOOK_BACK}, features={n_features})')
print(f'  X_val  : {X_val_lstm.shape}')
print(f'  X_test : {X_test_lstm.shape}')
print(f'  y_train: {y_train_lstm.shape}  (scaled Close values)')

# Verify no future leakage: X_train[0] corresponds to rows 0..look_back-1
np.testing.assert_allclose(X_train_lstm[0], scaled_train[0:LOOK_BACK], rtol=1e-5)
print('\nNo-future-data assertion passed.')

In [ ]:
# ============================================================
# Tabular features for Random Forest / Linear Regression
# ============================================================
# For each row i (starting at N_LAG_CLOSE):
#   - Last N_LAG_CLOSE raw Close values (lag 5, 4, 3, 2, 1)
#   - Yesterday's full scaled feature vector (no future data)
# Target: raw Close at row i

def build_tabular_features(df, feature_scaler, feature_cols, n_lag=N_LAG_CLOSE):
    scaled = feature_scaler.transform(df[feature_cols].values)
    close_raw = df['Close'].values
    X_list, y_list = [], []
    for i in range(n_lag, len(df)):
        lag_closes   = close_raw[i - n_lag : i]  # last N raw Close values
        prev_features = scaled[i - 1]            # yesterday's scaled features
        row = np.concatenate([lag_closes, prev_features])
        X_list.append(row)
        y_list.append(close_raw[i])              # raw next-day Close
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def get_tabular_feature_names(feature_cols, n_lag=N_LAG_CLOSE):
    lag_names  = [f'Close_lag_{n_lag-i}' for i in range(n_lag)]
    prev_names = [f'prev_{c}' for c in feature_cols]
    return lag_names + prev_names

X_train_tab, y_train_tab = build_tabular_features(df_train, feature_scaler, FEATURE_COLUMNS)
X_val_tab,   y_val_tab   = build_tabular_features(df_val,   feature_scaler, FEATURE_COLUMNS)
X_test_tab,  y_test_tab  = build_tabular_features(df_test,  feature_scaler, FEATURE_COLUMNS)

TAB_FEATURE_NAMES = get_tabular_feature_names(FEATURE_COLUMNS)

print(f'Tabular feature shapes:')
print(f'  X_train: {X_train_tab.shape}  X_val: {X_val_tab.shape}  X_test: {X_test_tab.shape}')
print(f'  y_train: {y_train_tab.shape}  (raw Close prices)')
print(f'  Feature columns ({len(TAB_FEATURE_NAMES)}): {TAB_FEATURE_NAMES[:6]} ...')

# The LSTM and tabular test sets differ by LOOK_BACK vs N_LAG_CLOSE offset.
# For a fair comparison, we align test targets to the same overlapping dates.
# LSTM test starts at df_test.index[LOOK_BACK], tabular at df_test.index[N_LAG_CLOSE].
# We use df_test.index[LOOK_BACK:] as the common evaluation window.
EVAL_OFFSET = LOOK_BACK  # start of comparable test period (largest offset)
print(f'\nCommon evaluation window: df_test.index[{EVAL_OFFSET}:]')
print(f'  ({len(df_test) - EVAL_OFFSET} overlapping test dates for all three models)')

---
## Section 7 — LSTM Model Development & Training

Architecture:
```
LSTM(128, return_sequences=True)
Dropout(0.2)
LSTM(64)
Dropout(0.2)
Dense(32, relu)
Dense(1)
```
Optimizer: Adam | Loss: MSE | Early stopping (patience=10, restore best weights)

In [ ]:
# ============================================================
# Build LSTM model
# ============================================================

def build_lstm_model(look_back, n_features):
    model = keras.Sequential([
        layers.Input(shape=(look_back, n_features)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ], name='stock_lstm')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mean_squared_error',
        metrics=['mae'],
    )
    return model

lstm_model = build_lstm_model(LOOK_BACK, n_features)
lstm_model.summary()

In [ ]:
# ============================================================
# Train LSTM
# ============================================================
print(f'Training LSTM on {len(X_train_lstm)} sequences...')
print(f'Validation on {len(X_val_lstm)} sequences (chronologically AFTER train).')
print(f'Early stopping: patience={PATIENCE}, restore_best_weights=True')
print()

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

lstm_history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)

epochs_run = len(lstm_history.history['loss'])
print(f'\nLSTM training complete: {epochs_run} epoch(s) run.')
print(f'Best val_loss: {min(lstm_history.history["val_loss"]):.6f}')

In [ ]:
# ============================================================
# LSTM Training vs Validation Loss Curve
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(lstm_history.history['loss'],     label='Train Loss', color='#2196F3', lw=2)
ax.plot(lstm_history.history['val_loss'], label='Val Loss',   color='#FF5722', lw=2)
ax.set_title('LSTM Training vs. Validation Loss (MSE)', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
loss_path = os.path.join(FIGURES_DIR, 'lstm_loss_curve.png')
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'LSTM loss curve saved → {loss_path}')

In [ ]:
# ============================================================
# LSTM — Generate test predictions
# ============================================================
lstm_test_scaled = lstm_model.predict(X_test_lstm, verbose=0)
# Inverse-transform using dedicated close_scaler
lstm_test_preds = close_scaler.inverse_transform(lstm_test_scaled).flatten()

# Actual test Close values aligned with LSTM output
# LSTM test starts at row LOOK_BACK of df_test
lstm_test_actual = df_test['Close'].iloc[LOOK_BACK:].values
lstm_test_dates  = df_test.index[LOOK_BACK:]

print(f'LSTM test predictions generated: {len(lstm_test_preds)} values')
print(f'Test dates: {lstm_test_dates[0].date()} → {lstm_test_dates[-1].date()}')
print(f'Sample — Actual: ${lstm_test_actual[0]:.2f}  Predicted: ${lstm_test_preds[0]:.2f}')

---
## Section 8 — Random Forest Regressor Development & Training

In [ ]:
# ============================================================
# Train Random Forest Regressor (n_estimators=200)
# ============================================================
print(f'Training Random Forest (n_estimators={RF_ESTIMATORS})...')
rf_model = RandomForestRegressor(
    n_estimators=RF_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_model.fit(X_train_tab, y_train_tab)
print('Random Forest training complete.')

# Generate test predictions
# Align with LSTM by using the same EVAL_OFFSET rows from df_test
# Tabular features start at N_LAG_CLOSE; LSTM at LOOK_BACK
# We slice X_test_tab so predictions align with lstm_test_dates
tab_offset = LOOK_BACK - N_LAG_CLOSE  # how many extra rows LSTM drops
X_test_tab_aligned = X_test_tab[tab_offset:]
y_test_tab_aligned = y_test_tab[tab_offset:]

rf_test_preds  = rf_model.predict(X_test_tab_aligned)
rf_test_actual = y_test_tab_aligned  # raw prices

print(f'RF test predictions: {len(rf_test_preds)} values')
print(f'Sample — Actual: ${rf_test_actual[0]:.2f}  Predicted: ${rf_test_preds[0]:.2f}')

---
## Section 9 — Linear Regression Development & Training

In [ ]:
# ============================================================
# Train Linear Regression baseline
# ============================================================
print('Training Linear Regression baseline...')
lr_model = LinearRegression()
lr_model.fit(X_train_tab, y_train_tab)
print('Linear Regression training complete.')

# Aligned test predictions
lr_test_preds  = lr_model.predict(X_test_tab_aligned)
lr_test_actual = y_test_tab_aligned  # same as RF (already aligned)

print(f'LR test predictions: {len(lr_test_preds)} values')
print(f'Sample — Actual: ${lr_test_actual[0]:.2f}  Predicted: ${lr_test_preds[0]:.2f}')

---
## Section 10 — Model Evaluation & Comparison

All metrics computed on the **original dollar price scale**.
All three models evaluated on the **same overlapping test dates**.

> ⚠️ These are actual computed metrics — not hardcoded reference values.

In [ ]:
# ============================================================
# Compute evaluation metrics (original price scale)
# ============================================================

def compute_metrics(y_true, y_pred, model_name=''):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# Verify all three models evaluated on the same actual prices
# lstm_test_actual, rf_test_actual, lr_test_actual should be identical
assert len(lstm_test_actual) == len(rf_test_actual) == len(lr_test_actual), \
    'Test set sizes differ — alignment error!'
np.testing.assert_allclose(lstm_test_actual, rf_test_actual, rtol=1e-5,
    err_msg='Actual values for LSTM vs RF differ — alignment error!')

lstm_metrics = compute_metrics(lstm_test_actual, lstm_test_preds, 'LSTM')
rf_metrics   = compute_metrics(rf_test_actual,   rf_test_preds,   'Random Forest')
lr_metrics   = compute_metrics(lr_test_actual,   lr_test_preds,   'Linear Regression')

all_metrics = [lstm_metrics, rf_metrics, lr_metrics]
metrics_df  = pd.DataFrame(all_metrics).set_index('Model')
metrics_df.columns = ['RMSE ($)', 'MAE ($)', 'R²']

print('='*60)
print('MODEL EVALUATION RESULTS (Test Period — Original Price Scale)')
print('='*60)
print(f'Test period: {lstm_test_dates[0].date()} → {lstm_test_dates[-1].date()}')
print(f'Test observations: {len(lstm_test_actual)}')
print()
display(metrics_df.round(4).style
    .highlight_min(axis=0, color='#c8e6c9')
    .format({'RMSE ($)': '${:.4f}', 'MAE ($)': '${:.4f}', 'R²': '{:.4f}'})
)
print()
print('Note: Metrics computed by the code — not copied from any reference report.')

In [ ]:
# ============================================================
# Model comparison bar chart
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'{TICKER} — Model Comparison (Test Period)', fontsize=14, fontweight='bold')

models  = metrics_df.index.tolist()
colors  = ['#2196F3', '#4CAF50', '#FF5722']
metrics_to_plot = ['RMSE ($)', 'MAE ($)', 'R²']

for ax, metric in zip(axes, metrics_to_plot):
    vals = metrics_df[metric].values
    bars = ax.bar(models, vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_xticklabels(models, rotation=20, ha='right')
    for bar, val in zip(bars, vals):
        fmt = f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                fmt, ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
comp_path = os.path.join(FIGURES_DIR, 'model_comparison.png')
plt.savefig(comp_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Comparison chart saved → {comp_path}')

---
## Section 11 — Prediction Visualisation & Feature Importance

In [ ]:
# ============================================================
# Actual vs. Predicted — all models on same test dates
# ============================================================
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(lstm_test_dates, lstm_test_actual, label='Actual',           color='black', lw=2.5)
ax.plot(lstm_test_dates, lstm_test_preds,  label='LSTM',             color='#2196F3', lw=1.5, ls='--')
ax.plot(lstm_test_dates, rf_test_preds,    label='Random Forest',    color='#4CAF50', lw=1.5, ls='--')
ax.plot(lstm_test_dates, lr_test_preds,    label='Linear Regression',color='#FF5722', lw=1.5, ls='--')

ax.set_title(f'{TICKER} — Actual vs. Predicted Closing Prices (Test Period)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Closing Price ($)')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.grid(alpha=0.3)
fig.tight_layout()

pred_path = os.path.join(FIGURES_DIR, 'actual_vs_predicted.png')
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Actual vs predicted chart saved → {pred_path}')

In [ ]:
# ============================================================
# Random Forest Feature Importance
# ============================================================
importances = rf_model.feature_importances_
feat_imp_dict = dict(sorted(zip(TAB_FEATURE_NAMES, importances), key=lambda x: x[1], reverse=True))

top_n = 15
top_names = list(feat_imp_dict.keys())[:top_n]
top_vals  = list(feat_imp_dict.values())[:top_n]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_names[::-1], top_vals[::-1], color='#4CAF50', alpha=0.85, edgecolor='white')
ax.set_title(f'{TICKER} — Random Forest Feature Importance (Top {top_n})', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()

fi_path = os.path.join(FIGURES_DIR, 'feature_importance.png')
plt.savefig(fi_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Feature importance chart saved → {fi_path}')

# Save feature importance as JSON
fi_json_path = os.path.join(ARTIFACT_DIR, 'feature_importance.json')
with open(fi_json_path, 'w') as f:
    json.dump(feat_imp_dict, f, indent=2)
print(f'Feature importance JSON saved → {fi_json_path}')

In [ ]:
# ============================================================
# Residual Analysis (LSTM)
# ============================================================
residuals = lstm_test_actual - lstm_test_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{TICKER} — LSTM Residual Analysis', fontsize=14, fontweight='bold')

# Time series residuals
axes[0].plot(lstm_test_dates, residuals, color='#9C27B0', lw=1)
axes[0].axhline(0, color='red', ls='--', lw=1)
axes[0].fill_between(lstm_test_dates, residuals, 0, alpha=0.2, color='#9C27B0')
axes[0].set_title('Residuals Over Time (Actual − Predicted)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Residual ($)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[0].grid(alpha=0.3)

# Histogram
axes[1].hist(residuals, bins=40, color='#9C27B0', alpha=0.75, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=1)
axes[1].axvline(residuals.mean(), color='blue', ls='--', lw=1, label=f'Mean={residuals.mean():.2f}')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
res_path = os.path.join(FIGURES_DIR, 'lstm_residuals.png')
plt.savefig(res_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Residual plot saved → {res_path}')
print(f'LSTM Residual stats — Mean: {residuals.mean():.4f}, Std: {residuals.std():.4f}')

---
## Section 12 — Save Trained Models, Scalers, Metrics & Charts

In [ ]:
# ============================================================
# Save LSTM model (.keras format — recommended for TF2)
# ============================================================
lstm_path = os.path.join(ARTIFACT_DIR, f'{TICKER}_lstm.keras')
lstm_model.save(lstm_path)
print(f'LSTM model saved → {lstm_path}')

# ============================================================
# Save Random Forest and Linear Regression (.joblib)
# ============================================================
rf_path = os.path.join(ARTIFACT_DIR, f'{TICKER}_random_forest.joblib')
lr_path = os.path.join(ARTIFACT_DIR, f'{TICKER}_linear_regression.joblib')
joblib.dump(rf_model, rf_path)
joblib.dump(lr_model, lr_path)
print(f'Random Forest saved → {rf_path}')
print(f'Linear Regression saved → {lr_path}')

# ============================================================
# Save scalers (.joblib)
# ============================================================
feat_scaler_path  = os.path.join(ARTIFACT_DIR, 'feature_scaler.joblib')
close_scaler_path = os.path.join(ARTIFACT_DIR, 'close_scaler.joblib')
joblib.dump(feature_scaler, feat_scaler_path)
joblib.dump(close_scaler,   close_scaler_path)
print(f'Feature scaler saved → {feat_scaler_path}')
print(f'Close scaler saved   → {close_scaler_path}')

In [ ]:
# ============================================================
# Save metadata (feature columns, config, training info)
# ============================================================
metadata = {
    'ticker'           : TICKER,
    'feature_cols'     : FEATURE_COLUMNS,
    'tab_feature_names': TAB_FEATURE_NAMES,
    'look_back'        : LOOK_BACK,
    'n_lag_close'      : N_LAG_CLOSE,
    'train_ratio'      : TRAIN_RATIO,
    'val_ratio'        : VAL_RATIO,
    'random_seed'      : RANDOM_SEED,
    'train_date_range' : [str(df_train.index[0].date()), str(df_train.index[-1].date())],
    'val_date_range'   : [str(df_val.index[0].date()),   str(df_val.index[-1].date())],
    'test_date_range'  : [str(df_test.index[0].date()),  str(df_test.index[-1].date())],
    'eval_window'      : [str(lstm_test_dates[0].date()), str(lstm_test_dates[-1].date())],
    'n_train_rows'     : len(df_train),
    'n_val_rows'       : len(df_val),
    'n_test_rows'      : len(df_test),
    'n_features'       : n_features,
    'lstm_epochs_run'  : epochs_run,
    'rf_estimators'    : RF_ESTIMATORS,
    'price_convention' : 'adjusted_close_auto_adjust_true',
    'created_at'       : datetime.utcnow().isoformat() + 'Z',
}

meta_path = os.path.join(ARTIFACT_DIR, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {meta_path}')
print()
print('Saved configuration:')
for k, v in metadata.items():
    if not isinstance(v, list) or len(str(v)) < 80:
        print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Save evaluation metrics CSV
# ============================================================
metrics_csv_path = os.path.join(ARTIFACT_DIR, 'metrics.csv')
metrics_df.to_csv(metrics_csv_path)
print(f'Metrics CSV saved → {metrics_csv_path}')
display(metrics_df.round(4))

# ============================================================
# Save test-period predictions CSV (for Streamlit app)
# ============================================================
test_preds_df = pd.DataFrame({
    'Actual'            : lstm_test_actual,
    'LSTM'              : lstm_test_preds,
    'Random Forest'     : rf_test_preds,
    'Linear Regression' : lr_test_preds,
}, index=lstm_test_dates)
test_preds_df.index.name = 'Date'

preds_csv_path = os.path.join(ARTIFACT_DIR, 'test_predictions.csv')
test_preds_df.to_csv(preds_csv_path)
print(f'Test predictions CSV saved → {preds_csv_path}')

In [ ]:
# ============================================================
# Verify saved models can be loaded and produce same predictions
# ============================================================
print('Verifying model save/load integrity...')

# Reload LSTM
lstm_reloaded = keras.models.load_model(lstm_path)
lstm_check = lstm_reloaded.predict(X_test_lstm[:5], verbose=0)
lstm_orig  = lstm_model.predict(X_test_lstm[:5],    verbose=0)
np.testing.assert_allclose(lstm_orig, lstm_check, rtol=1e-4)
print('  LSTM: reload check PASSED')

# Reload RF
rf_reloaded = joblib.load(rf_path)
rf_check    = rf_reloaded.predict(X_test_tab_aligned[:5])
rf_orig     = rf_model.predict(X_test_tab_aligned[:5])
np.testing.assert_allclose(rf_orig, rf_check, rtol=1e-4)
print('  Random Forest: reload check PASSED')

# Reload LR
lr_reloaded = joblib.load(lr_path)
lr_check    = lr_reloaded.predict(X_test_tab_aligned[:5])
lr_orig     = lr_model.predict(X_test_tab_aligned[:5])
np.testing.assert_allclose(lr_orig, lr_check, rtol=1e-4)
print('  Linear Regression: reload check PASSED')

# Reload scalers
feat_sc_reloaded  = joblib.load(feat_scaler_path)
close_sc_reloaded = joblib.load(close_scaler_path)
test_check = feat_sc_reloaded.transform(df_test[FEATURE_COLUMNS].values[:3])
test_orig  = feature_scaler.transform(df_test[FEATURE_COLUMNS].values[:3])
np.testing.assert_allclose(test_orig, test_check, rtol=1e-5)
print('  Scalers: reload check PASSED')

print()
print('All save/load integrity checks PASSED.')

---
## Section 13 — Generate Next Trading Day Prediction

Uses the exact same preprocessing pipeline as training:
1. Latest N features from `df_feat`
2. Apply **saved** feature scaler (no refitting)
3. Build 60-day LSTM sequence or tabular row
4. Inverse-transform with **saved** close scaler

In [ ]:
# ============================================================
# Inference helper functions
# ============================================================

def build_inference_sequence(df, feature_scaler, feature_cols, look_back=LOOK_BACK):
    """Build (1, look_back, n_features) array for LSTM inference."""
    if len(df) < look_back:
        raise ValueError(f'Need >= {look_back} rows, got {len(df)}')
    tail = df[feature_cols].iloc[-look_back:].values
    return feature_scaler.transform(tail)[np.newaxis, :, :].astype(np.float32)

def build_tabular_inference_row(df, feature_scaler, feature_cols, n_lag=N_LAG_CLOSE):
    """Build (1, n_lag + n_features) array for RF/LR inference."""
    if len(df) < n_lag:
        raise ValueError(f'Need >= {n_lag} rows, got {len(df)}')
    lag_closes   = df['Close'].values[-n_lag:]
    last_scaled  = feature_scaler.transform(df[feature_cols].iloc[-1:].values).flatten()
    row = np.concatenate([lag_closes, last_scaled]).astype(np.float32)
    return row.reshape(1, -1)

def next_trading_day(last_date):
    next_d = last_date + timedelta(days=1)
    while next_d.weekday() >= 5:
        next_d += timedelta(days=1)
    return next_d

print('Inference functions ready.')

In [ ]:
# ============================================================
# Generate next-trading-day predictions for all three models
# ============================================================
last_close = float(df_feat['Close'].iloc[-1])
last_date  = df_feat.index[-1]
pred_date  = next_trading_day(last_date)

print(f'Last observed date  : {last_date.date()}')
print(f'Last observed Close : ${last_close:.2f}')
print(f'Prediction target   : {pred_date.date()}')
print()

# --- LSTM ---
X_infer_lstm = build_inference_sequence(df_feat, feature_scaler, FEATURE_COLUMNS, LOOK_BACK)
lstm_pred_scaled = lstm_model.predict(X_infer_lstm, verbose=0)
lstm_next_close  = float(close_scaler.inverse_transform(lstm_pred_scaled)[0][0])

# --- Random Forest ---
X_infer_tab = build_tabular_inference_row(df_feat, feature_scaler, FEATURE_COLUMNS, N_LAG_CLOSE)
rf_next_close = float(rf_model.predict(X_infer_tab)[0])

# --- Linear Regression ---
lr_next_close = float(lr_model.predict(X_infer_tab)[0])

# ============================================================
# Display prediction results
# ============================================================
print('='*60)
print(f'NEXT-DAY PREDICTIONS FOR {TICKER}')
print(f'Predicted date: {pred_date.date()}')
print('='*60)

results = {
    'LSTM'              : lstm_next_close,
    'Random Forest'     : rf_next_close,
    'Linear Regression' : lr_next_close,
}

prediction_rows = []
for model_name, pred_price in results.items():
    delta_abs = pred_price - last_close
    delta_pct = (delta_abs / last_close) * 100
    direction = '▲' if delta_abs >= 0 else '▼'
    print(f'  {model_name:<20}: ${pred_price:.2f}  ({direction} {delta_abs:+.2f} / {delta_pct:+.2f}%)')
    prediction_rows.append({
        'Model'          : model_name,
        'Last Close ($)' : round(last_close, 2),
        'Predicted ($)'  : round(pred_price, 2),
        'Change ($)'     : round(delta_abs, 2),
        'Change (%)'     : round(delta_pct, 2),
        'Predicted Date' : str(pred_date.date()),
    })

print()
print('⚠️  Educational forecast only — not investment advice.')

# Save predictions to JSON
pred_output = {
    'ticker'       : TICKER,
    'last_date'    : str(last_date.date()),
    'last_close'   : round(last_close, 4),
    'predicted_date': str(pred_date.date()),
    'predictions'  : {k: round(v, 4) for k, v in results.items()},
    'generated_at' : datetime.utcnow().isoformat() + 'Z',
}
pred_json_path = os.path.join(ARTIFACT_DIR, 'next_day_prediction.json')
with open(pred_json_path, 'w') as f:
    json.dump(pred_output, f, indent=2)
print(f'Predictions saved → {pred_json_path}')

In [ ]:
# Display predictions as a formatted table
pred_df = pd.DataFrame(prediction_rows).set_index('Model')
display(pred_df.style
    .format({'Last Close ($)': '${:.2f}', 'Predicted ($)': '${:.2f}',
             'Change ($)': '${:+.2f}', 'Change (%)': '{:+.2f}%'})
    .applymap(lambda v: 'color: green' if isinstance(v, (int, float)) and v > 0 else
              ('color: red' if isinstance(v, (int, float)) and v < 0 else ''),
              subset=['Change ($)', 'Change (%)'])
)

---
## Section 14 — Final Verification & Summary

In [ ]:
# ============================================================
# Final verification: check all key artifacts exist
# ============================================================
print('='*60)
print('FINAL VERIFICATION CHECKLIST')
print('='*60)

def check_file(path, label):
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = f'✅ ({size:.1f} KB)' if exists else '❌ MISSING'
    print(f'  {status}  {label}')
    print(f'          {path}')
    return exists

checks = [
    (os.path.join(RAW_DIR, f'{TICKER}_daily.csv'),            'Raw CSV'),
    (processed_path,                                           'Processed CSV'),
    (lstm_path,                                                'LSTM model (.keras)'),
    (rf_path,                                                  'Random Forest (.joblib)'),
    (lr_path,                                                  'Linear Regression (.joblib)'),
    (feat_scaler_path,                                         'Feature scaler (.joblib)'),
    (close_scaler_path,                                        'Close scaler (.joblib)'),
    (meta_path,                                                'Metadata (metadata.json)'),
    (metrics_csv_path,                                         'Metrics (metrics.csv)'),
    (preds_csv_path,                                           'Test predictions CSV'),
    (fi_json_path,                                             'Feature importance JSON'),
    (pred_json_path,                                           'Next-day prediction JSON'),
    (os.path.join(FIGURES_DIR, 'eda_overview.png'),            'EDA chart'),
    (os.path.join(FIGURES_DIR, 'technical_indicators.png'),    'Technical indicators chart'),
    (os.path.join(FIGURES_DIR, 'lstm_loss_curve.png'),         'LSTM loss curve'),
    (os.path.join(FIGURES_DIR, 'model_comparison.png'),        'Model comparison chart'),
    (os.path.join(FIGURES_DIR, 'actual_vs_predicted.png'),     'Actual vs predicted chart'),
    (os.path.join(FIGURES_DIR, 'feature_importance.png'),      'Feature importance chart'),
    (os.path.join(FIGURES_DIR, 'lstm_residuals.png'),          'Residual plot'),
]

all_ok = all(check_file(path, label) for path, label in checks)
print()
if all_ok:
    print('All artifacts verified ✅')
else:
    print('Some artifacts missing ❌ — re-run the corresponding section.')

In [ ]:
# ============================================================
# Final summary
# ============================================================
print('='*60)
print(f'STOCK PRICE PREDICTION PIPELINE — COMPLETE SUMMARY')
print('='*60)
print(f'  Ticker          : {TICKER}')
print(f'  Data range      : {df_feat.index[0].date()} → {df_feat.index[-1].date()}')
print(f'  Total rows      : {len(df_feat)}')
print(f'  Features        : {n_features} ({FEATURE_COLUMNS})')
print(f'  Train/Val/Test  : {len(df_train)} / {len(df_val)} / {len(df_test)}')
print(f'  Eval window     : {len(lstm_test_actual)} overlapping test dates')
print()
print('  MODEL METRICS (Test Period — Original Dollar Scale):')
for _, row in metrics_df.iterrows():
    print(f"  {row.name:<22} RMSE=${row['RMSE ($)']:.4f}  MAE=${row['MAE ($)']:.4f}  R²={row['R²']:.4f}")
print()
print('  NEXT-DAY PREDICTIONS:')
print(f'  Last observed: ${last_close:.2f} on {last_date.date()}')
print(f'  Predicted date: {pred_date.date()}')
for model_name, price in results.items():
    print(f'  {model_name:<22}: ${price:.2f}')
print()
print('  ARTIFACTS LOCATION:', ARTIFACT_DIR)
print('  FIGURES LOCATION  :', FIGURES_DIR)
print()
print('⚠️  DISCLAIMER: Educational project only. Predictions are not investment advice.')
print('    Market prices are inherently uncertain. Past patterns ≠ future results.')
print('='*60)
print('Pipeline complete! To use the Streamlit app:')
print('  1. Copy the artifacts/ folder to your project directory')
print('  2. Run: streamlit run app.py')
print('  Or use ngrok in Colab: !pip install pyngrok && ngrok tunneling')